# P6a: Fig Engine envelope calculation

Measures sequential read bandwidth on the current Colab runtime and projects I/O, dequantization, and GEMM time for reference 4B, 8B, and 26B model shapes. Results are saved durably to Google Drive.

Run every cell in order. This notebook does not start P6b or any later stage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
REPO_URL = 'https://github.com/Harboria-Labs/littlefig.git'
BRANCH = 'research/p1-figmezo-verify'
REPO_DIR = '/content/littlefig'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Repository:', os.getcwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import psutil
vm = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'RAM: {vm.total/1e9:.2f} GB total, {vm.available/1e9:.2f} GB available')
print(f'/content disk: {disk.total/1e9:.2f} GB total, {disk.free/1e9:.2f} GB free')

In [ ]:
import json, pathlib, sys
RESULTS = '/content/drive/MyDrive/littlefig-p6a/envelope_v1_results.json'
pathlib.Path(RESULTS).parent.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, '-u', 'benchmark/experiment_envelope_v1.py',
       '--read-size-mib', '512', '--read-block-mib', '8', '--read-repeats', '3',
       '--results-path', RESULTS]
print('RUNNING:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)
assert pathlib.Path(RESULTS).exists(), f'Missing result: {RESULTS}'
print('Saved:', RESULTS)

In [ ]:
with open(RESULTS, encoding='utf-8') as f:
    result = json.load(f)
print(f"Measured conservative read bandwidth: {result['read_bandwidth']['conservative_gib_s']:.3f} GiB/s")
for name, row in result['models'].items():
    t = row['projected_forward_totals_s']
    print(f"{name}: I/O={t['io_s']:.3f}s, dequant={t['dequant_s']:.3f}s, GEMM={t['gemm_s']:.3f}s -> {row['bottleneck']}")
result